In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:90% !important;}
div.cell.code_cell.rendered{width:100%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:12pt;}
div.output {font-size:12pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:12pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:12px;}
</style>
"""))

<font size="5" color="black">ch04. tensorflow v1</font>

# 1. tensorflow v1


| **TensorFlow 1.x 구버전** | **TensorFlow 2.x 신버전** |
| :--- | :--- |
| `tf.placeholder()` | `tf.keras.Input()` 또는 함수 인자 사용 |
| `tf.Variable()` + `Session.run()` | `tf.Variable()` (즉시 연산 가능) |
| `sess.run(train_step, feed_dict={...})` | `tf.GradientTape()` 또는 `model.fit()` |
| **정적 그래프 (Define-and-Run)**<br>그래프 설계 후 `tf.Session()`에서 실행 | **즉시 실행 (Eager Execution / Define-by-Run)**<br>Python/NumPy처럼 코드 선언 즉시 결과 계산 |
| C++ 최적화 및 분산 환경 처리 유리 | 직관적인 디버깅 및 간결한 Keras API 통합 |
| `tf.placeholder`, `tf.Session`, `Graph` | `tf.keras`, `tf.GradientTape`, `@tf.function` |

In [3]:
import tensorflow.compat.v1 as tf
tf.disable_v2_behavior() # tensorflow v2 비활성화 및 v1 활성화
import numpy as np
import pandas as pd

Instructions for updating:
non-resource variables are not supported in the long term


## Tensorflow
- 데이터 흐름 그래프(Tensor 흐름을 나타내는 설계도)를 사용하는 수치 계산 라이브러리
- 구성 요소 : tensor(다차원 배열 데이터 구조), node(수치 연산 및 동작), edge(흐름 및 관계)
- `sess = tf.Session()`을 이용하여 실행 환경(세션) 생성
- `sess.run()`을 통해 결과 확인

### tensor 정의
- 상수 node, 변수 node, 연산 node 정의 = tensor 객체 정의

In [7]:
node1 = tf.constant('Hello, Tensorflow')
print(node1) # node : 설계도 정보 출력됨

Tensor("Const_1:0", shape=(), dtype=string)


### 세션 생성

In [5]:
sess = tf.Session()

### 실행

In [8]:
print(sess.run(node1)) # 문자열 출력의 경우, 바이너리로 출력됨
print(sess.run(node1).decode())

b'Hello, Tensorflow'
Hello, Tensorflow


### 연산 node

In [11]:
# 1. node 정의
node1 = tf.constant(10, dtype=tf.float16)
node2 = tf.constant(20, dtype=tf.float16) # 텐서의 연산에서, 각 노드의 타입은 같아야함

# 2. 연산 node 정의
node3 = tf.add(node1, node2)

# 3. 세션 생성
sess = tf.Session()

# 4. 실행
n1, n2, n3 = sess.run([node1, node2, node3])
print(n1, n2, n3)

10.0 20.0 30.0


### 타입 변경

In [13]:
node1 = tf.constant(np.array([10, 20, 30]), dtype=tf.int16)
node2 = tf.cast(node1, dtype=tf.float32)
sess = tf.Session()
print(sess.run([node1, node2]))

[array([10, 20, 30], dtype=int16), array([10., 20., 30.], dtype=float32)]


### 평균값 계산
- tf.reduce_mean()

In [18]:
data = np.array([1., 2, 3, 4]) # data의 타입이 정수일 경우, 계산 결과도 정수로 산출됨 → 정확한 결과 수집을 위해 실수 타입 설정 필요
m = tf.reduce_mean(data)
sess = tf.Session()
sess.run(m)

2.5

### 난수 발생
- tf.random_normal([shape]) : 평균 0, 표준편차 1인 shape 모양의 난수 배열 생성 (기본값 : float32)

In [20]:
w = tf.random_normal([1, 2])
sess = tf.Session()
sess.run(w)

array([[ 0.10996553, -0.438134  ]], dtype=float32)

### 변수 node

In [25]:
w = tf.Variable(tf.random_normal([1]))
sess = tf.Session()
sess.run(tf.global_variables_initializer()) # 설계도(그래프)에 선언된 모든 변수들의 초기값을 일괄 주입
sess.run(w)

array([-1.3487316], dtype=float32)

# 2. tf v1 회귀분석
- tensorflow v1을 이용한 회귀분석 구현

## 2.1 독립변수 x, 종속변수 y 각각 1개

### 설계도 정의

In [52]:
# 데이터셋 확보
x = np.array([1,2,3])
y = np.array([2,3,4])

In [53]:
# weight와 bias
w = tf.Variable(tf.random_normal([1]), name='weight')
b = tf.Variable(tf.random_normal([1]), name='bias')

In [54]:
# hat, hypothesis : numpy 배열을 출력
H = w * x + b

In [55]:
# cost function (손실함수 mse ; H-y 제곱의 평균)
cost = tf.reduce_mean(tf.square(H-y))

In [56]:
# optimizer를 통한 cost 최솟값 도출 시도
optimizer = tf.train.GradientDescentOptimizer(learning_rate=0.01)
train = optimizer.minimize(cost)

In [57]:
# 세션 생성
sess = tf.Session()

In [58]:
# w와 b 초기화
sess.run(tf.global_variables_initializer())

### 학습

In [59]:
for step in range(1, 5001):
    _, cost_val, w_val, b_val = sess.run([train, cost, w, b])
    if step%500==1:    
        print(f'{step} 번째 cost = {cost_val}, w = {w_val}, b = {b_val}')
print(f'{step} 번째 cost = {cost_val}, w = {w_val}, b = {b_val}')

1 번째 cost = 15.008213996887207, w = [-0.20243765], b = [0.10304333]
501 번째 cost = 0.0012347678421065211, w = [1.0407141], b = [0.9074477]
1001 번째 cost = 0.00011124855518573895, w = [1.0122207], b = [0.97221935]
1501 번째 cost = 1.0023602953879163e-05, w = [1.0036683], b = [0.99166113]
2001 번째 cost = 9.039927135745529e-07, w = [1.0011017], b = [0.9974958]
2501 번째 cost = 8.19419767594809e-08, w = [1.0003316], b = [0.9992461]
3001 번째 cost = 7.535347457121588e-09, w = [1.0001009], b = [0.9997715]
3501 번째 cost = 6.966113796913476e-10, w = [1.0000305], b = [0.99993044]
4001 번째 cost = 8.057909894887416e-11, w = [1.0000106], b = [0.99997663]
4501 번째 cost = 5.328478283606053e-11, w = [1.0000087], b = [0.9999811]
5000 번째 cost = 5.328478283606053e-11, w = [1.0000087], b = [0.9999811]


In [61]:
w_, b_ = sess.run([w[0], b[0]])
w_, b_

(1.0000087, 0.9999811)

In [62]:
def predict(x):
    return x*w_ + b_

In [63]:
predict(5)

6.000024616718292

## 2.2 placeholder
- placeholder : 외부 데이터(X, y)를 연산 그래프 내부로 주입하기 위한 노드

In [68]:
X = tf.placeholder(dtype=np.float32)
H = w_*X + b_
sess = tf.Session()
sess.run([H, X], {X:2.5})

[3.5000029, array(2.5, dtype=float32)]

In [70]:
sess.run(H, {X: np.array([2.5, 3, 3.5])})

array([3.5000029, 4.000007 , 4.5000114], dtype=float32)

In [73]:
x_data = np.array([1,2,3])
y_data = np.array([2,3,4])

# placeholder 노드 설정
x = tf.placeholder(dtype=tf.float32)
y = tf.placeholder(dtype=tf.float32)

w = tf.Variable(tf.random_normal([1]), name='weight')
b = tf.Variable(tf.random_normal([1]), name='bias')

H = w * x + b

cost = tf.reduce_mean(tf.square(H-y))

train = tf.train.GradientDescentOptimizer(learning_rate=0.005).minimize(cost)

sess = tf.Session()

sess.run(tf.global_variables_initializer())

for step in range(1, 5001):
    _, cost_val, w_val, b_val = sess.run([train, cost, w, b], feed_dict={x:x_data, y:y_data})
    if step%500==1:    
        print(f'{step} 번째 cost = {cost_val}, w = {w_val}, b = {b_val}')
print(f'{step} 번째 cost = {cost_val}, w = {w_val}, b = {b_val}')

1 번째 cost = 2.73805832862854, w = [0.50601244], b = [0.47765782]
501 번째 cost = 0.002821569098159671, w = [1.0616195], b = [0.85992414]
1001 번째 cost = 0.0008475554059259593, w = [1.0337721], b = [0.923228]
1501 번째 cost = 0.00025459457538090646, w = [1.0185096], b = [0.9579234]
2001 번째 cost = 7.647915481356904e-05, w = [1.0101447], b = [0.9769385]
2501 번째 cost = 2.2975234969635494e-05, w = [1.0055605], b = [0.98736]
3001 번째 cost = 6.904263500473462e-06, w = [1.0030482], b = [0.99307114]
3501 번째 cost = 2.07647644856479e-06, w = [1.0016719], b = [0.99620044]
4001 번째 cost = 6.254926461224386e-07, w = [1.0009179], b = [0.99791455]
4501 번째 cost = 1.8903369891631883e-07, w = [1.000505], b = [0.998854]
5000 번째 cost = 5.745467390738668e-08, w = [1.000278], b = [0.9993677]


In [78]:
# 예측하기
# feed_dict : 외부에서 정의한 입력값(tf.placeholder)에 실제 데이터를 주입(Feed)
sess.run(H, feed_dict={x:2.5})

array([25.198706], dtype=float32)

In [79]:
sess.run(H, feed_dict={x:np.array([2.5, 3.5])})

array([25.198706, 35.352383], dtype=float32)

## 2.3 scale
- scale이 다른 데이터의 회귀분석

In [76]:
# scale 조정 X
x_data = np.array([1,2,5,8,10])
y_data = np.array([5,15,68,80,95])

x = tf.placeholder(dtype=tf.float32)
y = tf.placeholder(dtype=tf.float32)

w = tf.Variable(tf.random_normal([1]), name='weight')
b = tf.Variable(tf.random_normal([1]), name='bias')

H = w * x + b

cost = tf.reduce_mean(tf.square(H-y))

# optimizer = tf.train.GradientDescentOptimizer(learning_rate=0.1)
# 손실값(Cost)이 발산하여 nan이 출력
optimizer = tf.train.GradientDescentOptimizer(learning_rate=0.001)
train = optimizer.minimize(cost)

sess = tf.Session()

sess.run(tf.global_variables_initializer())

for step in range(1, 5001):
    _, cost_val, w_val, b_val = sess.run([train, cost, w, b], feed_dict={x:x_data, y:y_data})
    if step%500==1:    
        print(f'{step} 번째 cost = {cost_val}, w = {w_val}, b = {b_val}')
print(f'{step} 번째 cost = {cost_val}, w = {w_val}, b = {b_val}')

1 번째 cost = 6645.6279296875, w = [-1.7289046], b = [-0.92173535]
501 번째 cost = 79.2746353149414, w = [10.069734], b = [0.43604714]
1001 번째 cost = 79.2139663696289, w = [10.092944], b = [0.2641853]
1501 번째 cost = 79.18051147460938, w = [10.110175], b = [0.13658647]
2001 번째 cost = 79.16209411621094, w = [10.12297], b = [0.04185055]
2501 번째 cost = 79.15193176269531, w = [10.13247], b = [-0.02848748]
3001 번째 cost = 79.1463394165039, w = [10.139527], b = [-0.08070952]
3501 번째 cost = 79.14327239990234, w = [10.144763], b = [-0.11948429]
4001 번째 cost = 79.14155578613281, w = [10.148646], b = [-0.14827175]
4501 번째 cost = 79.140625, w = [10.151535], b = [-0.16964789]
5000 번째 cost = 79.14007568359375, w = [10.153677], b = [-0.18548773]


### scale 조정
- 모든 데이터를 일정 범위 내로 조정
    - normalization (정규화) $= \frac{X - X_{\min}}{X_{\max} - X_{\min}}$ : 데이터의 범위를 0과 1 사이로 변환
        - 라이브러리 : sklearn.preprocessing.MinMaxScaler
    - standardization (표준화) $= \frac{X - X_{\mu}}{X_{\sigma}}$ : 데이터의 평균${\mu}$이 0이고 표준편차${\sigma}$가 1인 정규분포 형태로 변환
        - 라이브러리 : sklearn.preprocessing.StandardScaler

In [82]:
# 라이브러리를 사용하지 않은 정규화
x_data = np.array([1,2,5,8,10])
y_data = np.array([5,15,68,80,95])

norm_scaled_x_data = (x_data - x_data.min())/(x_data.max() - x_data.min())
norm_scaled_y_data = (y_data - y_data.min())/(y_data.max() - y_data.min())

In [91]:
# 라이브러리를 활용한 정규화
from sklearn.preprocessing import MinMaxScaler, StandardScaler

# MinMaxScaler : 2차원 배열([Samples, Features])을 입력으로 받음
x_data = np.array([1,2,5,8,10]).reshape(-1,1)
y_data = np.array([5,15,68,80,95]).reshape(-1,1)

scaler_x = MinMaxScaler()
scaler_y = MinMaxScaler()

norm_scaled_x_data = scaler_x.fit_transform(x_data)
norm_scaled_y_data = scaler_y.fit_transform(y_data)

np.column_stack([x_data, norm_scaled_x_data, y_data, norm_scaled_y_data])

array([[ 1.        ,  0.        ,  5.        ,  0.        ],
       [ 2.        ,  0.11111111, 15.        ,  0.11111111],
       [ 5.        ,  0.44444444, 68.        ,  0.7       ],
       [ 8.        ,  0.77777778, 80.        ,  0.83333333],
       [10.        ,  1.        , 95.        ,  1.        ]])

In [104]:
# 라이브러리를 사용하지 않은 표준화
x_data = np.array([1,2,5,8,10])
y_data = np.array([5,15,68,80,95])

std_scaled_x_data = (x_data - x_data.mean())/(x_data.std())
std_scaled_y_data = (y_data - y_data.mean())/(y_data.std())
print(np.column_stack([x_data, std_scaled_x_data]))
print()
print(np.column_stack([y_data, std_scaled_y_data]))

[[ 1.         -1.22474487]
 [ 2.         -0.93313895]
 [ 5.         -0.05832118]
 [ 8.          0.81649658]
 [10.          1.39970842]]

[[ 5.         -1.32373476]
 [15.         -1.04563922]
 [68.          0.42826713]
 [80.          0.76198177]
 [95.          1.17912508]]


In [105]:
# 라이브러리를 활용한 정규화
from sklearn.preprocessing import MinMaxScaler, StandardScaler

# StandardScaler : 2차원 배열([Samples, Features])을 입력으로 받음
x_data = np.array([1,2,5,8,10]).reshape(-1,1)
y_data = np.array([5,15,68,80,95]).reshape(-1,1)

# 범위 혼용 방지를 위한 객체 개별 생성
scaler_x = StandardScaler()
scaler_y = StandardScaler()

std_scaled_x_data = scaler_x.fit_transform(x_data)
std_scaled_y_data = scaler_y.fit_transform(y_data)

print(np.column_stack([x_data, std_scaled_x_data]))
print()
print(np.column_stack([y_data, std_scaled_y_data]))

[[ 1.         -1.22474487]
 [ 2.         -0.93313895]
 [ 5.         -0.05832118]
 [ 8.          0.81649658]
 [10.          1.39970842]]

[[ 5.         -1.32373476]
 [15.         -1.04563922]
 [68.          0.42826713]
 [80.          0.76198177]
 [95.          1.17912508]]


In [121]:
# scale 조정 O

x = tf.placeholder(dtype=tf.float32)
y = tf.placeholder(dtype=tf.float32)

w = tf.Variable(tf.random_normal([1]), name='weight')
b = tf.Variable(tf.random_normal([1]), name='bias')

H = w * x + b

cost = tf.reduce_mean(tf.square(H-y))

optimizer = tf.train.GradientDescentOptimizer(learning_rate=0.001)
train = optimizer.minimize(cost)

sess = tf.Session()

sess.run(tf.global_variables_initializer())

for step in range(1, 10001):
    _, cost_val, w_val, b_val = sess.run([train, cost, w, b], feed_dict={x:norm_scaled_x_data, y:norm_scaled_y_data})
    if step%3000==1:    
        print(f'{step} 번째 cost = {cost_val}, w = {w_val}, b = {b_val}')
print(f'{step} 번째 cost = {cost_val}, w = {w_val}, b = {b_val}')

1 번째 cost = 0.6871427297592163, w = [-0.6282209], b = [0.29027903]
3001 번째 cost = 0.08019883930683136, w = [0.32841823], b = [0.41757253]
6001 번째 cost = 0.02717474475502968, w = [0.6742592], b = [0.2352578]
9001 번째 cost = 0.014071369543671608, w = [0.84610826], b = [0.14448941]
10000 번째 cost = 0.012470634654164314, w = [0.8813824], b = [0.12585798]


### scale 복원
- inverse_transform()

In [111]:
scaler_x.inverse_transform(std_scaled_x_data)

array([[ 1.],
       [ 2.],
       [ 5.],
       [ 8.],
       [10.]])

In [107]:
scaler_y.inverse_transform(std_scaled_y_data)

array([[ 5.],
       [15.],
       [68.],
       [80.],
       [95.]])

## 2.4 독립변수 x 3개, 종속변수 y 1개